<center>
<img src="https://laelgelcpublic.s3.sa-east-1.amazonaws.com/lael_50_years_narrow_white.png.no_years.400px_96dpi.png" width="300" alt="LAEL 50 years logo">
<h3>APPLIED LINGUISTICS GRADUATE PROGRAMME (LAEL)</h3>
</center>
<hr>

# Corpus Linguistics - Study 1 - Phase 1 - Leticia

This document aims to gather data for project planning.

## Determine speakers and define the AI speaker

Define the AI speaker as `S0999` since the highest speaker id is `S0691`.

In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET

xml_dir = Path("corpus/bnc2014spoken-xml/spoken/untagged")
speakers = set()

for xml_file in xml_dir.glob("*.xml"):
    root = ET.parse(xml_file).getroot()
    for elem in root.findall(".//list_speakers"):
        if elem.text:
            speakers.update(elem.text.split())

print(", ".join(sorted(speakers)))


S0001, S0002, S0003, S0004, S0005, S0006, S0007, S0008, S0009, S0010, S0011, S0012, S0013, S0014, S0015, S0016, S0017, S0018, S0019, S0020, S0021, S0022, S0023, S0024, S0025, S0026, S0027, S0028, S0029, S0030, S0031, S0032, S0033, S0034, S0035, S0036, S0037, S0038, S0039, S0040, S0041, S0042, S0043, S0044, S0045, S0046, S0047, S0048, S0049, S0050, S0051, S0052, S0053, S0054, S0055, S0056, S0057, S0058, S0059, S0060, S0061, S0062, S0063, S0064, S0065, S0066, S0067, S0068, S0069, S0070, S0071, S0072, S0073, S0074, S0075, S0076, S0077, S0078, S0079, S0080, S0081, S0082, S0083, S0084, S0085, S0086, S0087, S0088, S0089, S0090, S0091, S0092, S0093, S0094, S0095, S0096, S0097, S0098, S0099, S0100, S0101, S0102, S0103, S0104, S0105, S0106, S0107, S0108, S0109, S0110, S0113, S0114, S0115, S0116, S0117, S0118, S0119, S0120, S0121, S0122, S0123, S0124, S0125, S0126, S0127, S0128, S0129, S0130, S0131, S0132, S0133, S0134, S0135, S0136, S0137, S0138, S0139, S0140, S0141, S0142, S0143, S0144, S0145,

## Detemine the speaker with the highest number of turns in each conversation

In [7]:
import pandas as pd
from collections import Counter

rows = []

for xml_file in xml_dir.glob("*.xml"):
    root = ET.parse(xml_file).getroot()
    text_id = root.get("id", xml_file.stem)
    turns = root.findall(".//body/u")
    total_turns = len(turns)
    speaker_counts = Counter(
        u.get("who") for u in turns
        if u.get("who")
    )

    if speaker_counts:
        top_talker, top_talker_turns = sorted(
            speaker_counts.items(),
            key=lambda item: (-item[1], item[0])
        )[0]
    else:
        top_talker, top_talker_turns = None, 0

    rows.append(
        {
            "text_id": text_id,
            "total_turns": total_turns,
            "top_talker": top_talker,
            "top_talker_turns": top_talker_turns,
        }
    )

turns_df = pd.DataFrame(rows).sort_values("text_id").reset_index(drop=True)
turns_df.to_json("top_talkers.ndjson", orient="records", lines=True)

turns_df


,text_id,total_turns,top_talker,top_talker_turns
0,S23A,4041,S0094,1203
1,S24A,127,S0261,63
2,S24D,455,S0653,200
3,S24E,709,S0520,283
4,S263,3160,S0590,1086
...,...,...,...,...
1246,SZVB,1463,S0517,730
1247,SZVC,798,S0325,398
1248,SZW4,479,S0510,238
1249,SZXQ,838,S0058,411


## Detemine the speaker with the highest number of turns overall

In [3]:
peak_conversation = turns_df.loc[turns_df["total_turns"].idxmax(), ["text_id", "total_turns"]]
top_top_talker = turns_df.loc[turns_df["top_talker_turns"].idxmax(), ["text_id", "top_talker", "top_talker_turns"]]

print(
    f"Conversation with peak number of turns: {peak_conversation['text_id']} ({peak_conversation['total_turns']} turns)")
print(
    f"Top top talker: {top_top_talker['top_talker']} in conversation {top_top_talker['text_id']} ({top_top_talker['top_talker_turns']} turns)")


Conversation with peak number of turns: SUVQ (16574 turns)
Top top talker: S0198 in conversation SUVQ (4601 turns)


## Determine the conversation with the least number of turns

In [6]:
least_conversation = turns_df.loc[turns_df["total_turns"].idxmin(), ["text_id", "total_turns", "top_talker"]]

print(
    f"Conversation with the least number of turns: {least_conversation['text_id']} "
    f"({least_conversation['total_turns']} turns); top talker: {least_conversation['top_talker']}"
)


Conversation with the least number of turns: SEV9 (67 turns); top talker: S0202


## Determine the total number of turns produced by the top talkers

In [4]:
total_top_talker_turns = turns_df["top_talker_turns"].sum()
print(f"Total number of turns produced by the top talkers: {total_top_talker_turns}")


Total number of turns produced by the top talkers: 510697


## Import the dataset into a DataFrame

Write the `01_import_bnc2014sp.py` programme according to the following specifications:

1. Load the NDJSON file containing top-talker information into the `df_top_talkers` DataFrame.
   - Required columns: `text_id`, `top_talker`

2. Parse all XML files in `corpus/bnc2014spoken-xml/spoken/untagged/`.

3. Create `df_bnc2014sp_header` with one row per XML file.
   - Include `text_id`
   - Include one column for each direct child tag of `<header>`, excluding `<speakerInfo>`
   - Normalize text content
   - Store missing values as null

4. Create the speaker information datasets from `<speakerInfo>`:

   a. Create `df_bnc2014sp_speaker_info_occurrences` with:
   - `text_id`
   - `speaker_id`
   - one column per corresponding speaker metadata tag

   b. Create `df_bnc2014sp_speakers` as a speaker database with:
   - `speaker_id`
   - one consolidated value per speaker metadata field

   During parsing:
   - add new speakers when first encountered
   - if an existing speaker is found, compare metadata with previous occurrences
   - log any mismatches
   - preserve occurrence-level records
   - keep one consolidated row per `speaker_id` in the speaker database

   Sort the speaker database by `speaker_id` ascending.

5. Create `df_bnc2014sp_conversation` with one row per utterance in `<body>`.
   - Include `text_id`
   - Include a sequential turn index
   - Include `speaker_id`
   - Include utterance text
   - Include relevant utterance attributes
   - Add `top_talker` with value `Yes` if `speaker_id` equals the top talker for that `text_id`, otherwise `No`

6. Save the DataFrames in `corpus/01_bnc2014sp_dataset/` as NDJSON files using their names without the `df_` prefix.
   - Use record-oriented NDJSON format
   - UTF-8 encoding

7. Log processing progress, parse issues, and speaker metadata mismatches.